# Lesson 04 - ടൂൾ ഉപയോഗ ഡിസൈൻ പാറ്റേൺ

ഈ പാഠത്തിൽ നിങ്ങൾക്ക് മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് (Python) ഉപയോഗിച്ച് AI ഏജൻന്റുകളുടെ **ടൂൾ ഉപയോഗ** ഡിസൈൻ പാറ്റേൺ മനസിലാക്കാം. നാം ഉൾപ്പെടുന്നവ:

- `@tool` ഡെക്കറേറ്റർഉം ടൈപ്പ് ചെയ്ത പാരാമീറ്ററുകളും ഉപയോഗിച്ച് ഫംഗ്ഷൻ ടൂളുകൾ നിർവചിക്കൽ
- മോഡലിന് ഓരോ ടൂളും എന്താ പ്രവർത്തിക്കുന്നതെന്ന് അറിയാൻ ടൂൾ സ്കീമകൾ നൽകൽ
- `approval_mode` ഉപയോഗിച്ച് ടൂൾ എക്‌സിക്യൂഷൻ നിയന്ത്രിക്കൽ
- Pydantic മോഡലുകളും `response_format` ഉപയോഗിച്ച് **സംരചിത ഔട്ട്പുട്ട്** മടക്കിവരുത്തൽ

സാധാരണ സാഹചര്യം ഒരു **യാത്ര ബുക്കിംഗ് ഏജന്റ്** ആണ്, ഇത് ഡെസ്റ്റിനേഷനുകൾ പരിശോധിക്കാനും, ലഭ്യതയൊരുക്കാനും, ഫ്ലൈറ്റ് വിവരങ്ങൾ എടുക്കാനും കഴിയും.


## സെറ്റപ്പ്


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ഡിസൈനെറ്ററോടുള്ള ടൂളുകൾ നിർവചിക്കുക

`@tool` ഡിസൈനെറ്റർ ഒരു സാധാരണ Python ഫംഗ്ഷൻ ഒരു ഏജന്റ് വിളിക്കാവുന്ന ടൂളായി മാറാൻ സഹായിക്കുന്നു.  
പ്രധാനപ്പെട്ട കാര്യങ്ങൾ:

- **ഡോക്സ്ട്രിംഗ്** മോഡൽ കാണുന്ന ടൂൾ വിവരണമാണ്.
- **ടൈപ്പ് അനോട്ടേഷനുകൾ** (`Annotated` ഉൾപ്പെടെ വിവരണങ്ങളോടുകൂടി) ടൂൾ സ്‌കീമ നിർവചിക്കുന്നു.
- `approval_mode` നടത്തിയുകൊണ്ടിരിക്കുന്ന ഓരോ കോളിനും മുൻപ് ഉപയോക്താവ് അംഗീകാരം നൽകേണ്ടതുണ്ടോ എന്ന് നിയന്ത്രിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ഒന്നിലധികം ടൂളുകളുള്ള ഏജന്റ് സൃഷ്ടിക്കുന്നത്

ഉപയോക്താവിന്റെ ಪ್ರಶ್ನയ്ക്ക് ഉത്തരമാകാൻ മോഡൽ ആവശ്യപ്പെടുന്ന ഏത് ടൂളും വിളിക്കാനാകും വിധം മൂന്ന് ടൂളുകളും ക്ലയന്റിന് കൈമാറുക.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ടൂളുകളോടെ ഘടിത ഔട്ട്‌പുട്ട്

`response_format` ഒരു പൈഡാന്റിക് മോഡലായി സെറ്റ് ചെയ്താൽ, ഏജന്റ് സ്വതന്ത്രരൂപത്തിലുള്ള ടെക്സ്റ്റ് പകരം ശ്രദ്ധാപൂർവം ടൈപ്പുചെയ്ത JSON ഒബ്‌ജക്റ്റ് മടങ്ങിവരുത്താൻ നിർബന്ധിതനാകും. താഴെ പ്രവർത്തിക്കുന്ന കോഡ് ഫലത്തെ പ്രോഗ്രാമാറ്റിക്കായി ഉപയോഗിക്കേണ്ടപ്പോഴാണിത് ഉപകാരപ്രദം.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ടൂൾ അനുമതി മാതൃകകൾ

`@tool` ലെ `approval_mode` പാരാമീറ്റർ ടൂൾ കോൾ പ്രവർത്തിപ്പിക്കാനുള്ള മുൻപ് മനുഷ്യന്റെ അനുമതി ആവശ്യമാണ് എങ്കിൽ നിയന്ത്രിക്കുന്നു:

| മാതൃക | പെരുമാറ്റം |
|---|---|
| `"never_require"` | ടൂൾ സ്വയം ക്രിയാത്മകമാകും — ഉപയോക്തൃ സ്ഥിരീകരണം ആവശ്യമില്ല. |
| `"always_require"` | ഓരോ കോഴും പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പായി ഉപയോക്താവിന്റെയും അനുമതി വേണം. |

സൈഡ്-ഇഫക്ട്ൾ ഉള്ള ടൂളുകൾക്ക് `"always_require"` ഉപയോഗിക്കുക (ഉദാ: വിമാന ടിക്കറ്റുകൾ بک ചെയ്യുക, ക്രഡിറ്റ് കാർഡ് ചാർജ് ചെയ്യുക) അതിനാൽ ഒരു മനുഷ്യൻ പ്രക്രിയയിൽ തുടരും.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. ടൂൾ സ്കീമയായി സേവനം ചെയ്യുന്ന ടൈപ്പുചെയ്ത പാരാമീറ്ററുകളും ഡോക്‌സ്റ്റ്രിംഗുകളും ഉള്ള `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് **ടൂളുകൾ നിർവചിക്കുന്നതും**.
2. ഏജന്റിന് സങ്കീർണ്ണമായ ചോദ്യങ്ങളോട് ഉത്തരമറിയിക്കാനാകുന്നതിനു **ചില ടൂളുകൾ ഒരുമിച്ചുപയോഗിക്കുന്നത്**.
3. ഒരു പൈഡാന്റിക് മോഡൽ `response_format` ആയി പാസ്സ് ചെയ്ത് **ഘടനാത്മക ഔട്ട്‌പുട്ട് നൽകുന്നത്**.
4. അഭ്യർത്ഥിച്ച പ്രവർത്തനങ്ങൾക്ക് മനുഷ്യൻ ഇടപെടുന്നവരായിരിക്കാൻ **`approval_mode` ഉപയോഗിച്ചുകൊണ്ട് ടൂൾ അംഗിയിൽ നിയന്ത്രണം**.

ഈ പരമ്പരകൾ വിശ്വസനീയവും പ്രൊഡക്ഷൻ-റെഡി ആയ ഏജന്റുകൾ നിർമ്മിക്കുന്നതിനുള്ള അടിസ്ഥാന ഘടകങ്ങളായാണ്, അവ സുരക്ഷിതമായി പുറംമണ്ഡല സംവിധാനങ്ങളുമായി ഇടപഴകാൻ കഴിയുന്നവ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
